<a href="https://colab.research.google.com/github/sarahmdias19/AI-ML-For-Fluids/blob/main/Introduction_to_PyTorch_Day1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
torch.__version__

'2.11.0+cu128'

In [ ]:
import torch
torch.cuda.is_available()

True

# Scalars, Vectors, Matrices and Tensors

In [ ]:
tensor0d = torch.tensor(1) # Create a 0d tensor (scalar) from a Python integer
tensor1d = torch.tensor([1,2,3]) # Create a 1d tensor (vector) from a Python list
tensor2d = torch.tensor([[1,2],[3,4]]) # Create a 1d tensor (matrix/array) from a nested Python list
tensor3d = torch.tensor([[[1,2],[3,4]],[[5,6],[7,8]]]) # Create a 3d tensor from a nested Python list

## Tensor data types

In [ ]:
print(tensor1d.dtype)

torch.int64


In [ ]:
float_vec = torch.tensor([1.0, 2.0, 3.0])
print(float_vec.dtype)

torch.float32


In [ ]:
floatvec = tensor1d.to(torch.float32)
print(floatvec.dtype)

torch.float32


## Common PyTorch Tensor Operations

In [ ]:
tensor2d = torch.tensor([[1,2,3],[4,5,6]])
tensor2d

tensor([[1, 2, 3],
        [4, 5, 6]])

In [ ]:
tensor2d.shape

torch.Size([2, 3])

In [ ]:
tensor2d.reshape(3,2)

tensor([[1, 2],
        [3, 4],
        [5, 6]])

In [ ]:
tensor2d.view(3,2)

tensor([[1, 2],
        [3, 4],
        [5, 6]])

In [ ]:
tensor2d.T

tensor([[1, 4],
        [2, 5],
        [3, 6]])

In [ ]:
tensor2d.matmul(tensor2d.T)

tensor([[14, 32],
        [32, 77]])

In [ ]:
tensor2d @ tensor2d.T

tensor([[14, 32],
        [32, 77]])

## Models as computation graphs

In [ ]:
import torch.nn
import torch.nn.functional as F

x1 = torch.tensor([1.1]) # input feature
w1 = torch.tensor([2.2]) # weight parameter
b = torch.tensor([0.0]) # bias unit
y1 = torch.tensor([1.0]) # true label

z1 = w1*x1 + b # net input
a = torch.sigmoid(z1) # activation & ouput

loss = F.binary_cross_entropy(a, y1)
print(loss)


tensor(0.0852)


## Automatic differentiation (Autograd)

In [ ]:
import torch.nn
import torch.nn.functional as F
from torch.autograd import grad

x1 = torch.tensor([1.1]) # input feature
w1 = torch.tensor([2.2], requires_grad=True) # weight parameter
b = torch.tensor([0.0], requires_grad=True) # bias unit
y1 = torch.tensor([1.0]) # true label

z1 = w1*x1 + b # net input
a = torch.sigmoid(z1) # activation & ouput

loss = F.binary_cross_entropy(a, y1)
#print(loss)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)


(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [ ]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


## Exercise
Compute the derivative using PyTorch
$$y = 3x^3$$

## Mechanics of Automatic Differentiation
PyTorch computes exact partial derivatives from numbers using Reverse-Mode Automatic Differentiation (Autograd) rather than numerical approximations. It accomplishes this by tracking mathematical operations dynamically on a directed acyclic graph (DAG) and evaluating hardcoded analytical derivative formulas using the chain rule.

### The Core Mechanism
PyTorch does not use symbolic algebra (like SymPy) nor does it use finite difference approximations (f(x+h) - f(x) / h). Instead, it breaks complex equations down into a sequence of basic primitive operations (such as addition, multiplication, or sine functions).

### The Three Step Process
1. **Dynamic Graph Construction (Forward Pass)**
When you create a tensor with `requires_grad=True`, PyTorch flags it for tracking. As you perform mathematical operations using these tensors, PyTorch dynamically builds a computational graph in the background.
+ Every node in this graph is an operator (e.g., `Mul`, `Add`, `Pow`).
+ Every edge represents the data flowing through.
+ During this pass, PyTorch caches the intermediate numerical values because they are required later to compute the final derivatives.

2. **Built-In Derivative Lookups**
PyTorch developers have pre-programmed the exact calculus derivative formulas for every basic function inside its source engine (the derivatives.yaml file).
+ If you compute $y = x^2$, PyTorch already knows that the derivative formula is $2x$.
+ If you compute $y = \sin(x)$, PyTorch knows the derivative is $\cos(x)$.
+ Instead of deriving these formulas at runtime, it simply plugs your cached numerical values into these hardcoded derivative formulas

3. **Accumulation via the Chain Rule (Backward Pass)**
When you trigger `.backward()` on your final output (usually a scalar loss value), PyTorch traverses the computational graph in reverse. It multiplies the local derivatives of each node together using the calculus chain rule.
$$\frac{\partial \text{Output}}{\partial x} = \frac{\partial \text{Output}}{\partial y} \cdot \frac{\partial y}{\partial x}$$

The final numerical results of these multiplications are then stored directly inside the `.grad` attribute of your original input tensors.

#### Step-by-Step Code Example


In [ ]:
import torch

# 1. Initialize input numerical values and enable tracking
x = torch.tensor(3.0, requires_grad=True)
y = torch.tensor(4.0, requires_grad=True)

# 2. Forward pass: PyTorch computes the output (43.0) and stores the graph
z = 3 * (x ** 2) + (y ** 2)

# 3. Backward pass: Evaluates hardcoded formulas at these exact points
# Local derivative wrt x is 6x. Evaluated at x=3: 6 * 3 = 18
# Local derivative wrt y is 2y. Evaluated at y=4: 2 * 4 = 8
z.backward()

# Access the calculated partial derivatives
print(x.grad)  # Outputs: tensor(18.)
print(y.grad)  # Outputs: tensor(8.)

tensor(18.)
tensor(8.)
